# Trend Tracker — Notebook 03: Insights Generation

Builds feature matrices and runs all Phase 1 analytical work on the clean
corpus from `01_preprocess.ipynb`. `make_vec`, `add_bin`, and `group_key` —
all imported from `utils.py` — are shared across Steps 1, 3, and 4, which is
the main reason those three live together.

| Step | What | Key output |
|------|------|------------|
| 1 | Build TF-IDF matrices | `features/X_*.npz`, `vocab/vocab_*.csv` |
| 2 | Quality CP2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `nmf_weights.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |


## Setup

In [1]:
import sys, json, re, unicodedata
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from statistics import mean
import time as _time
import openai as _openai

import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.decomposition import NMF
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH

ROOT = Path(".")
sys.path.insert(0, str(ROOT))
from utils import (
    load_cfg, tokens_to_str, make_vec, add_bin, group_key, quality_report,
    build_run_output_path, get_run_date, get_llm_client, build_project_topic_bridge,
    slugify_group_value, canonicalize_filter_spec, get_filter_fields_key, get_run_id,
    apply_filters, ensure_warning_file, append_warning, write_json, artifact_meta,
    start_stage_manifest, finalize_stage_manifest, build_pipeline_manifest,
    load_essay_snippet_lookup, get_warning_count, get_effective_support_volume_scale
)

CFG = load_cfg()
ct = CFG["tfidf"]
client = get_llm_client()

raw_df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")
print(f"Loaded {len(raw_df):,} projects  |  {raw_df.shape[1]} columns")

Loaded 2,244,916 projects  |  46 columns


---
## Parameters

All tunable values live here. Edit this cell; do not edit parameter references in downstream cells.


In [2]:
# ── All parameters from params.yaml — edit there, not here ──────────────
GROUPBY_FIELD       = CFG["analysis"]["group_by"]
GROUP_DESCRIPTION   = CFG["analysis"]["group_descriptions"].get(
                          GROUPBY_FIELD,
                          f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose data")
MIN_SHARED          = CFG["analysis"]["nmf_min_shared"]
MIN_COVERAGE        = CFG["analysis"]["min_coverage"]
N_COMPONENTS        = CFG["nmf"]["n_components"]
CAT_TFIDF_TOP_N     = CFG["analysis"]["cat_tfidf_top_n"]
N_REPRESENTATIVE    = CFG["llm"]["n_representative_snippets"]
TOP_TERMS_IN_PROMPT = CFG["llm"]["top_terms_in_prompt"]
REVIEW_GROUP        = CFG["analysis"]["review_group"]
EXCLUDE_GROUPS      = CFG["analysis"]["exclude_groups"]
MAX_WORKERS         = CFG["analysis"]["synthesis_max_workers"]
LABELING_MAX_WORKERS = CFG["analysis"].get("labeling_max_workers", MAX_WORKERS)
MODEL_LABELING      = CFG["models"]["labeling"]
MODEL_SYNTHESIS     = CFG["models"]["synthesis"]

FILTER_LOGIC = CFG["analysis"].get("filter_logic", "and")
FILTERS      = CFG["analysis"].get("filters", [])
FILTER_SPEC  = canonicalize_filter_spec(FILTER_LOGIC, FILTERS)
FILTER_FIELDS_KEY = get_filter_fields_key(FILTERS)

df, filter_summary = apply_filters(raw_df, FILTER_LOGIC, FILTERS)
if filter_summary["no_rows_after_filter"]:
    raise ValueError("No rows remain after applying analysis filters.")

RUN_DATE = get_run_date()
RUN_ID   = get_run_id(GROUPBY_FIELD, FILTER_SPEC)

def OUT(subdir, fname):
    return build_run_output_path(
        subdir=subdir,
        fname=fname,
        groupby_field=GROUPBY_FIELD,
        run_date=RUN_DATE,
        run_id=RUN_ID,
    )

WARNINGS_PATH = OUT("metadata", "warnings_03.jsonl")
ensure_warning_file(WARNINGS_PATH)

FILTER_SPEC_PATH = OUT("metadata", "filter_spec.json")
FILTER_SUMMARY_PATH = OUT("metadata", "filter_summary.json")
write_json(FILTER_SPEC_PATH, {
    "schema_version": "v1",
    "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD,
    "filter_fields_key": FILTER_FIELDS_KEY,
    "filter_logic": FILTER_LOGIC,
    "filters": FILTERS,
})
write_json(FILTER_SUMMARY_PATH, {
    "schema_version": "v1",
    "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD,
    **filter_summary,
})

STAGE_MANIFEST = start_stage_manifest(
    stage_name="03_insights_generation",
    notebook_file="03_insights_generation_v0.1.ipynb",
    run_id=RUN_ID,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
)

bins       = CFG["analysis"].get("bins", [])
group_cols = [GROUPBY_FIELD, "bin"] if bins else [GROUPBY_FIELD]

print(f"RUN_ID           = {RUN_ID}")
print(f"GROUPBY_FIELD    = {GROUPBY_FIELD!r}")
print(f"FILTER_FIELDS_KEY= {FILTER_FIELDS_KEY}")
print(f"Filtered rows    = {len(df):,} / {len(raw_df):,}")
print(f"Output path      = OUTPUTS/runs/{GROUPBY_FIELD}/{RUN_DATE}/{RUN_ID}/")

RUN_ID           = 20260402_124534_miami_geo_group_45fd548b
GROUPBY_FIELD    = 'miami_geo_group'
FILTER_FIELDS_KEY= miami_geo_group
Filtered rows    = 197,762 / 2,244,916
Output path      = OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_124534_miami_geo_group_45fd548b/


---
## Step 1 — Build TF-IDF Matrices

One vectorizer fit per n-gram range on the full corpus. Trigrams are persisted
even if unused in Phase 1 analysis — Phase 2+ inherits them without re-extraction.


In [3]:
# Cache once — reused by Cells 10 and 12
docs = df["tokens"].apply(tokens_to_str).tolist()
specs = {"X_unigram": (1,1), "X_bigram": (2,2), "X_unigram_bigram": (1,2)}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec            = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name]     = vec
    sz, nnz        = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz/(sz[0]*sz[1]):.3f}")

for name, vec in vecs.items():
    (pd.DataFrame({"feature": vec.get_feature_names_out(), "idf": vec.idf_})
       .sort_values("idf")
       .to_csv(OUT("vocab", f"vocab_{name}.csv"), index=False))

feat_dir = OUT("features", "_anchor").parent
feat_dir.mkdir(parents=True, exist_ok=True)
for name, X in matrices.items():
    sp.save_npz(feat_dir / f"{name}.npz", X.tocsr())

META_COLS = [c for c in [
    "project_id", "project_category", "posted_date", "funded_date",
    "metro_type_at_time_of_posting", "grade_band", "theme_version",
    "posted_year", "posted_year_quarter", "school_is_low_income"
] if c in df.columns]
meta = df[META_COLS].reset_index(drop=True)
try:
    meta.to_parquet(feat_dir / "meta.parquet", index=False)
except ImportError:
    meta.to_csv(feat_dir / "meta.csv", index=False)

assert matrices["X_unigram"].shape[0] == len(df)

  X_unigram           : shape=(197762, 6889)  sparsity=0.991
  X_bigram            : shape=(197762, 236321)  sparsity=1.000
  X_unigram_bigram    : shape=(197762, 243210)  sparsity=1.000
  X_trigram           : shape=(197762, 115132)  sparsity=1.000


---
## Step 2 — Quality Report: Checkpoint 2


In [4]:
qr2 = quality_report(df, label="cp2", matrices=matrices,
                     save_path=OUT("quality", "quality_cp2.json"))


=======================================================  [cp2]
  Projects : 197,762
  Tok/proj : min=12  p50=69  max=102
  Vocab    : 7,574 unique tokens
  X_unigram           : shape=[197762, 6889]  sparsity=0.991
  X_bigram            : shape=[197762, 236321]  sparsity=1.000
  X_unigram_bigram    : shape=[197762, 243210]  sparsity=1.000
  X_trigram           : shape=[197762, 115132]  sparsity=1.000
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']



---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [5]:
top_n     = CAT_TFIDF_TOP_N
min_proj  = CFG["analysis"]["min_group_projects"]

df_work   = add_bin(df, bins) if bins else df.copy()
df_work   = df_work.reset_index(drop=True)
all_docs  = (df_work["tokens"].apply(tokens_to_str).tolist() if bins else docs)
vec_cat   = make_vec(ct["min_df"], ct["max_df"], tuple(ct["ngram_range"]))
X_full    = vec_cat.fit_transform(all_docs)
feat      = vec_cat.get_feature_names_out()
idf_vals  = vec_cat.idf_

def cat_tfidf_slice(idx):
    """Score category slice by index against the rest of the corpus."""
    rest_idx  = df_work.index.difference(idx)
    X_cat     = X_full[idx.tolist()]
    X_rest    = X_full[rest_idx.tolist()]
    cat_prev  = (X_cat  > 0).mean(axis=0).A1
    rest_prev = (X_rest > 0).mean(axis=0).A1 if len(rest_idx) else np.zeros(len(feat))
    tf        = X_cat.mean(axis=0).A1
    return (pd.DataFrame({
                "token":         feat,
                "tf":            tf,
                "idf":           idf_vals,
                "tfidf":         tf * idf_vals,
                "prevalence":    cat_prev,
                "contrast":      cat_prev - rest_prev,
                "project_count": (X_cat > 0).sum(axis=0).A1.astype(int),
            }).nlargest(top_n, "tfidf"))

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue
    kd  = group_key(keys, group_cols)
    top = cat_tfidf_slice(sub.index)
    for col, val in kd.items():
        top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(f"No groups met min_proj={min_proj} threshold — lower min_group_projects in params.yaml")
cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)
print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)


160 rows  |  8 groups


,miami_geo_group,token,tf,idf,tfidf,prevalence,contrast,project_count
0,"Broward County, FL",florida,0.007722,4.972271,0.038396,0.108891,0.094816,1079
1,"Broward County, FL",free reduce,0.009477,3.576689,0.033895,0.187506,0.117366,1858
2,"Broward County, FL",reduce,0.009887,3.273203,0.032363,0.211929,0.114698,2100
3,"Broward County, FL",title,0.010814,2.627233,0.028410,0.287012,0.095319,2844
4,"Broward County, FL",price,0.006929,4.098661,0.028399,0.121102,0.080006,1200
5,"Broward County, FL",reduce price,0.006783,4.146650,0.028129,0.117267,0.078194,1162
6,"Broward County, FL",lunch,0.009943,2.827194,0.028111,0.249167,0.092965,2469
7,"Broward County, FL",free,0.010251,2.727644,0.027962,0.265718,0.092663,2633
8,"Broward County, FL",price lunch,0.006537,4.244162,0.027744,0.110102,0.074856,1091
9,"Broward County, FL",teach,0.010764,2.404104,0.025877,0.307902,0.065606,3051


---
## Step 4 — NMF Topic Discovery

NMF is fit independently per category so the dominant axis in one category
does not suppress signal in others. Topics are candidates for human review —
not stable theme definitions (that is Phase 2).

The weights DataFrame records which projects load most strongly on each topic;
Step 5 uses it to select representative snippets by NMF weight rather than
randomly.


In [6]:
cn    = CFG["nmf"]
n_rep = N_REPRESENTATIVE
min_proj = CFG["analysis"]["min_group_projects"]

def nmf_one(docs):
    """Fit NMF on one slice. Returns (topics_df, W) or (None, None)."""
    vec = make_vec(ct["min_df"], ct["max_df"], tuple(ct.get("ngram_range", [1, 1])))
    X   = vec.fit_transform(docs)
    if X.shape[1] < N_COMPONENTS:
        return None, None
    model = NMF(
        n_components=N_COMPONENTS,
        random_state=cn["random_seed"],
        init="nndsvd",
        max_iter=cn["max_iter"],
    )
    W     = model.fit_transform(X)
    vocab = vec.get_feature_names_out()
    rows  = []
    for i, comp in enumerate(model.components_):
        idx = comp.argsort()[::-1][:cn["top_words"]]
        rows.append({
            "topic_id": i,
            "top_terms": vocab[idx].tolist(),
            "top_weights": comp[idx].tolist(),
        })
    return pd.DataFrame(rows), W

all_topics, all_weights = [], []
df_work = add_bin(df, bins) if bins else df.copy()
df_work = df_work.reset_index(drop=True)

NMF_GROUPS_SKIPPED = []
NMF_GROUPS_FAILED = []

for keys, sub in df_work.groupby(group_cols, observed=True):
    kd = group_key(keys, group_cols)
    group_value = kd[GROUPBY_FIELD]

    if len(sub) < min_proj:
        NMF_GROUPS_SKIPPED.append(str(group_value))
        continue

    group_docs = sub["tokens"].apply(tokens_to_str).tolist()
    pids = sub["project_id"].tolist()

    try:
        topics, W = nmf_one(group_docs)
    except Exception as e:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF failed for group '{group_value}'",
            context={"group": group_value, "error": str(e)},
        )
        continue

    if topics is None or W is None:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF skipped group '{group_value}' because the slice could not produce topics",
            context={"group": group_value},
        )
        continue

    for col, val in kd.items():
        topics[col] = val
    all_topics.append(topics)

    for tid in range(W.shape[1]):
        order = W[:, tid].argsort()[::-1]
        for rank, idx in enumerate(order):
            all_weights.append({
                **kd,
                "topic_id": tid,
                "project_id": pids[idx],
                "weight": float(W[idx, tid]),
                "rank": rank,
            })

if not all_topics:
    raise RuntimeError(
        f"No groups produced NMF topics — check N_COMPONENTS={N_COMPONENTS} vs vocab size, "
        f"or lower min_group_projects"
    )

topics_df  = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)
topics_df.to_csv(OUT("analysis", "nmf_topics.csv"), index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)
print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")

threshold = CFG["analysis"]["topic_assignment_threshold"]
project_topic_bridge_df = build_project_topic_bridge(weights_df, GROUPBY_FIELD, threshold)
project_topic_bridge_df.to_csv(OUT("analysis", "project_topic_bridge.csv"), index=False)
print(f"Bridge: {len(project_topic_bridge_df):,} project-topic assignments (topic_share >= {threshold})")

topics_df.head(6)

/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 400 reached. Increase it to improve convergence.
  warnings.warn(


160 topics across 8 groups
Bridge: 151,785 project-topic assignments (topic_share >= 0.3)


,topic_id,top_terms,top_weights,miami_geo_group
0,0,"[read, book, love, grade, level, reader, inter...","[0.9858499334316961, 0.7935885668791663, 0.537...","Broward County, FL"
1,1,"[do standardize, help do, next project, hallwa...","[0.41537885717980555, 0.4126689379753493, 0.40...","Broward County, FL"
2,2,"[choice decision, decision live, world choice,...","[0.39821547561638343, 0.39821547561638343, 0.3...","Broward County, FL"
3,3,"[achieve succeed, succeed inspiration, call ea...","[0.378405050434774, 0.3783633694151843, 0.3783...","Broward County, FL"
4,4,"[motivate hand, various challenge, florida cla...","[0.36289361119394087, 0.3526367633777557, 0.33...","Broward County, FL"
5,5,"[live forward, avenue live, successful avenue,...","[0.3643577426309868, 0.3643577426309868, 0.364...","Broward County, FL"


In [7]:
# Cross-group universal themes — terms appearing in NMF topics across many groups
if REVIEW_GROUP is not None:
    _eligible = topics_df[group_cols[0]].unique().tolist()
    if REVIEW_GROUP not in _eligible:
        raise ValueError(f"REVIEW_GROUP={REVIEW_GROUP!r} not found. Eligible: {_eligible}")
else:
    REVIEW_GROUP = df_work.groupby(GROUPBY_FIELD).size().idxmax()

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i+1:]:
        if ci == cj:
            continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE:
        continue
    if not any(terms <= prior for prior in seen):
        deduped.append({
            "theme": ", ".join(sorted(terms)[:5]),
            "n_groups": len(cats),
            "categories": sorted(cats),
        })
        seen.append(terms)

cross_group_df = pd.DataFrame(deduped).reset_index(drop=True)
cross_group_df.to_csv(OUT("analysis", "cross_group_themes.csv"), index=False)
cross_group_df

,theme,n_groups,categories
0,"book, grade, level, library, love",6,"[Broward County, FL, Houston, TX, Miami, FL, O..."
1,"chair, comfortable, desk, flexible, flexible seat",6,"[Broward County, FL, Houston, TX, Los Angeles,..."
2,"marker, material, organize, paper, pencil",6,"[Broward County, FL, Las Vegas, NV, Los Angele..."


---
## Step 5 — LLM Topic Labeling

One API call per topic on compressed input only — never raw essay text at scale.
Representative snippets are chosen by NMF weight (not random).

Prompt field limits (`top_unigrams`, `top_bigrams`, `top_nmf_terms`) are applied
as item counts. Parse failures are stored with enough metadata to debug later.


In [8]:
SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a JSON object, no preamble, no markdown fences.\n\n"
    "Token naming conventions you may see in topic terms:\n"
    "  __framing_[name]__ = a rhetorical framing signal injected during preprocessing "
    "(e.g. __framing_barrier_removal__ means essays removing obstacles to participation)\n"
    "  __cat_[name]__ = a subject matter category token (e.g. __cat_genetics__)\n"
    "  __sub_[name]__ = a subcategory token\n"
    "These are meaningful signals — incorporate them into your label and description.\n\n"
    "Labeling rules:\n"
    "- Prefer the most specific defensible label.\n"
    "- Preserve concrete signals such as named programs, vendors, pedagogies, student "
    "populations, classroom use cases, or rhetorical framing when present.\n"
    "- Do not collapse topics into broad umbrella labels like technology, literacy, "
    "engagement, classroom supplies, or social-emotional learning if the evidence "
    "supports a narrower interpretation.\n"
    "- When framing tokens are present, treat them as first-class evidence about how "
    "teachers are positioning the request, not just background metadata.\n"
    "- If a topic is mixed, say what subthemes are colliding rather than forcing an "
    "overly neat label.\n\n"
    "coherence_flag definitions:\n"
    "  coherent   — terms and representative snippets point clearly to one theme, "
    "population, or use case.\n"
    "  mixed      — two or more distinguishable subthemes are colliding in this topic; "
    "describe what they are in notes.\n"
    "  redundant  — this topic's terms and snippets substantially duplicate another "
    "topic in the same group, not merely overlap in subject area.\n"
    "  unclear    — terms are too scattered or generic to support a defensible label; "
    "the topic does not resolve to a coherent signal."
)
PROMPT = (
    f"Group ({GROUPBY_FIELD}): {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"
    "Instructions:\n"
    "- proposed_label should be concrete and specific, ideally naming the request or "
    "intervention, the population or context, and the framing when supported.\n"
    "- description should explain what makes this topic distinct from nearby generic topics.\n"
    "- If the topic appears broad, identify the narrower mechanism, use case, or population "
    "if the evidence supports it.\n"
    "- If the evidence is genuinely mixed, preserve that ambiguity instead of inventing a "
    "falsely precise label.\n\n"
    f"Return a JSON object with exactly these keys:\n"
    f"  {GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes\n"
    'coherence_flag must be one of: "coherent", "mixed", "redundant", "unclear" '
    "(definitions provided in system instructions)"
)

pid_text = df.set_index("project_id")["tokens"].apply(lambda t: " ".join(t[:40]))

def build_input(t_row):
    terms = t_row["top_terms"]
    key_cols = [GROUPBY_FIELD] + (["bin"] if "bin" in t_row.index else [])
    mask = weights_df["topic_id"] == t_row["topic_id"]
    for col in key_cols:
        mask &= weights_df[col] == t_row[col]
    rep_pids = (
        weights_df[mask]
        .sort_values("weight", ascending=False)["project_id"]
        .tolist()[:N_REPRESENTATIVE]
    )
    n_uni = TOP_TERMS_IN_PROMPT
    n_bi  = max(2, TOP_TERMS_IN_PROMPT // 2)
    n_nmf = TOP_TERMS_IN_PROMPT
    return {
        "group_value": t_row[GROUPBY_FIELD],
        "topic_id": int(t_row["topic_id"]),
        "bin_line": f"\nBin: {t_row['bin']}" if "bin" in t_row.index and pd.notna(t_row.get("bin")) else "",
        "unigrams":  ", ".join([x for x in terms if " " not in x][:n_uni]),
        "bigrams":   ", ".join([x for x in terms if " " in  x][:n_bi]),
        "nmf_terms": ", ".join(terms[:n_nmf]),
        "snippets":  "\n".join(f"- {pid_text.get(p, '')}" for p in rep_pids),
    }

def _make_label_error(inp, raw_text, code, error_text=None):
    return {
        "raw": raw_text,
        "parse_error": True,
        "error_code": code,
        "error": error_text,
        "model": MODEL_LABELING,
        "timestamp": datetime.now().isoformat(),
        GROUPBY_FIELD: inp["group_value"],
        "topic_id": inp["topic_id"],
    }

def _label_with_retry(inp, max_retries=3):
    text = ""
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_LABELING,
                messages=[
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": PROMPT.format(**inp)},
                ],
            )
            text = resp.choices[0].message.content.strip()
            obj = json.loads(text)
            obj["model"] = MODEL_LABELING
            obj["timestamp"] = datetime.now().isoformat()
            return obj
        except json.JSONDecodeError as e:
            if attempt < max_retries:
                _time.sleep(2 ** attempt)
                continue
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "LABELING_PARSE_FAILURE",
                f"Topic labeling JSON parse failure for {inp['group_value']} / topic {inp['topic_id']}",
                context={"group": inp["group_value"], "topic_id": inp["topic_id"], "error": str(e)},
            )
            return _make_label_error(inp, text, "LABELING_PARSE_FAILURE", str(e))
        except (_openai.RateLimitError, _openai.APITimeoutError) as e:
            if attempt < max_retries:
                _time.sleep(2 ** attempt)
            else:
                append_warning(
                    WARNINGS_PATH,
                    "03_insights_generation",
                    "LABELING_API_FAILURE",
                    f"Topic labeling failed after retries for {inp['group_value']} / topic {inp['topic_id']}",
                    context={"group": inp["group_value"], "topic_id": inp["topic_id"], "error": str(e)},
                )
                return _make_label_error(inp, text or str(e), "LABELING_API_FAILURE", str(e))
        except Exception as e:
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "LABELING_API_FAILURE",
                f"Topic labeling failed for {inp['group_value']} / topic {inp['topic_id']}",
                context={"group": inp["group_value"], "topic_id": inp["topic_id"], "error": str(e)},
            )
            return _make_label_error(inp, text or str(e), "LABELING_API_FAILURE", str(e))

topic_inputs = [build_input(t) for _, t in topics_df.iterrows()]

def _norm_group_value(value):
    return str(value or "").strip().casefold()

def _safe_topic_id(value):
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return -1

topic_order = {
    (_norm_group_value(inp["group_value"]), int(inp["topic_id"])): i
    for i, inp in enumerate(topic_inputs)
}

results = []
with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
    futures = {executor.submit(_label_with_retry, inp): inp for inp in topic_inputs}
    for future in as_completed(futures):
        inp = futures[future]
        obj = future.result()
        results.append(obj)
        print(f"  {inp['group_value']} / topic {inp['topic_id']} → {obj.get('proposed_label', '?')}")

for r in results:
    norm_key = (_norm_group_value(r.get(GROUPBY_FIELD, "")), _safe_topic_id(r.get("topic_id", -1)))
    if norm_key not in topic_order:
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "LABELING_SORT_KEY_MISMATCH",
            f"Label result did not match deterministic sort key for {r.get(GROUPBY_FIELD, '')} / topic {r.get('topic_id', '')}",
            context={"group": r.get(GROUPBY_FIELD, ""), "topic_id": r.get("topic_id", "")},
        )

results = sorted(
    results,
    key=lambda r: topic_order.get(
        (_norm_group_value(r.get(GROUPBY_FIELD, "")), _safe_topic_id(r.get("topic_id", -1))),
        10**9
    )
)

with open(OUT("analysis", "llm_topic_labels.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)
print(f"\n{len(results)} labels saved")

  Broward County, FL / topic 7 → Free/low-income lunch support and basic supplies for food-insecure families (Title I framing)
  Broward County, FL / topic 4 → Broward-area __framing_[neighborhood_pride]__ elementary classrooms requesting tools/materials to motivate diverse (free/reduced lunch) learners in grade-level mornings
  Broward County, FL / topic 2 → Income-qualification framing for 4th-grade social studies/ELA instruction materials and access (free/reduced lunch, district low parent income)
  Broward County, FL / topic 3 → Holiday community care and encouragement framing for low-income/pre-K students (__framing_[none]__)
  Broward County, FL / topic 0 → Early elementary literacy intervention centered on independent reading (___framing_joy_possibility_appeal___) using leveled books/reader centers
  Broward County, FL / topic 1 → ___framing_hallway_stop_next_project__ academic support and standardization for elementary autism/cluster classes (standardize test; pull program; aca

In [9]:
# Keep only successfully parsed labeling results in labels_df for downstream use
parse_errors = [r for r in results if r.get("parse_error")]
if parse_errors:
    print(f"WARNING: {len(parse_errors)} topics failed JSON parse — excluded from synthesis:")
    for e in parse_errors:
        print(f"  {e.get(GROUPBY_FIELD, '?')} / topic {e.get('topic_id', '?')}")

labels_df = pd.DataFrame([r for r in results if not r.get("parse_error")])

assert GROUPBY_FIELD in labels_df.columns, (
    f"labels_df missing '{GROUPBY_FIELD}' — re-run labeling for this groupby field"
)

all_groups = {str(r.get(GROUPBY_FIELD, "")) for r in results if str(r.get(GROUPBY_FIELD, "")).strip()}
labeled_groups = set(labels_df[GROUPBY_FIELD].astype(str).unique()) if not labels_df.empty else set()
LABELING_FAILED_GROUPS = sorted(all_groups - labeled_groups)

labels_df.groupby("coherence_flag").size().rename("count").to_frame()
labels_df[[GROUPBY_FIELD, "topic_id", "proposed_label", "coherence_flag", "description"]]

,miami_geo_group,topic_id,proposed_label,coherence_flag,description
0,"Broward County, FL",0,Early elementary literacy intervention centere...,coherent,This topic focuses on improving early reading ...
1,"Broward County, FL",2,Income-qualification framing for 4th-grade soc...,coherent,This topic centers on fourth graders (often EL...
2,"Broward County, FL",3,Holiday community care and encouragement frami...,coherent,This topic clusters around holiday/seasonal cl...
3,"Broward County, FL",4,Broward-area __framing_[neighborhood_pride]__ ...,coherent,Projects in local elementary schools (noted as...
4,"Broward County, FL",5,__cat_social_studies__-framed classroom cultur...,coherent,This topic centers on explicitly stated classr...
...,...,...,...,...,...
155,"San Diego, CA",Topic 10,__framing_basic_sufficiency____framing_teacher...,coherent,This topic centers on keeping classrooms stock...
156,miami_geo_group,9,__framing_teacher_as_caregiver__ counseling-an...,coherent,This topic centers on social-emotional develop...
157,miami_geo_group,10,Tampa kindergarten Renaissance/early-reader st...,coherent,This topic centers on kindergarten requests ti...
158,miami_geo_group,14,Military-connected students’ music instruction...,coherent,This topic centers on providing music educatio...


In [10]:
# ── SYNTHESIS — cross-group + per-group loops ─────────────────────────────

def clean_label(text):
    return re.sub(r'__[a-z_]+__\s*', '', str(text)).strip()

def build_topic_lines(df, groupby_field, group=None):
    if group is not None:
        df = df[df[groupby_field] == group]
    return "\n".join(
        f"  {row[groupby_field]} | topic {row.topic_id} | "
        f"label: {clean_label(row.proposed_label)} | "
        f"coherence: {row.coherence_flag} | "
        f"description: {clean_label(row.description)}"
        for _, row in df.iterrows()
    )

SYNTHESIS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You write precise, insight-driven briefings for internal strategy audiences. "
    "You do not pad your answers or state the obvious.\n\n"
    "Some topic labels may reference rhetorical framing signals (e.g. 'barrier removal "
    "framing', 'chronic scarcity language') — these come from injected tokens that "
    "categorize how teachers write, not just what they request. Treat these as "
    "meaningful signals about teacher rhetorical strategy.\n\n"
    "Prioritize nuance over novelty. Prefer fine-grained distinctions, concrete "
    "mechanisms, specific populations, and framing differences over broad thematic "
    "summaries.\n\n"
    "Grounding: when writing Framing Differences findings, the contrast must be "
    "traceable to explicit language in the topic label or description — not inferred "
    "from the topic's general subject matter or implied by category context. If you "
    "cannot quote or closely paraphrase specific label language to establish both sides "
    "of a contrast, do not make the finding."
)

topic_lines = build_topic_lines(labels_df, GROUPBY_FIELD)

SYNTHESIS_PROMPT_CROSS_GROUP = f"""
Below is a list of NMF topics discovered from teacher project request essays on DonorsChoose,
grouped by {GROUPBY_FIELD} ({GROUP_DESCRIPTION}).
Each topic represents a cluster of essays with similar language, framing, and request patterns.
{topic_lines}

Your task: identify the most specific, decision-useful patterns in this topic landscape.

Rules:
- Every finding must be grounded in specific topic labels, descriptions, or coherence flags
  from the list above.
- Do not introduce concepts, themes, or absences that cannot be traced to an actual topic.
- If you cannot name the specific group and topic, do not make the finding.
- Prefer fine-grained distinctions over broad summaries.
- Skip anything predictable given the group name.
- Skip the free/reduced lunch poverty framing — it appears everywhere and is already known.
- Skip generic engagement/motivation language unless the topic evidence makes it unusually specific.
- If fewer than two strong findings exist for a section, write one or write none. Do not pad.

Focus on:
1. RECURRING INTERNAL SPLITS — cases where the same subtheme distinction recurs independently
   inside multiple groups. Only flag a split if it appears in at least two groups with
   meaningfully similar internal logic. Name the groups and specific topics each time.

2. CROSS-GROUP SIGNATURES — a specific program, vendor, pedagogy, student need, or rhetorical
   frame that shows up as a recognizable topic in multiple groups. Name every group it appears
   in and explain why the pattern is meaningful.

3. FRAMING DIFFERENCES — cases where similar underlying needs are presented through meaningfully
   different rhetorical frames, populations, or classroom contexts. Explain what changes across
   groups and why that difference matters.

4. NOTABLE ABSENCES — a topic you would expect given the internal logic of a group's other
   topics, but that is missing. To qualify, name at least two specific topics already present
   in the same group that make the gap conspicuous from within. Do not reason from other groups.

5. REQUEST TRIGGERS AND PITCH STRATEGIES — patterns in why teachers say they are requesting and
   how they frame those reasons for a donor audience. Only use explicit topic evidence.

Formatting: use plain numbered section headers (e.g. '1. SUBTHEME DISTINCTIONS').
Do not use markdown headers (##). Do not use bullet points.

Format as five labeled sections. Under each, write 2-5 findings as short paragraphs.
Be specific — name the group and topic label every time. Do not use bullet points.
""".strip()

PER_GROUP_INSTRUCTIONS = """
Your task: identify the most specific, decision-useful patterns within this group.

Rules:
- Every finding must be grounded in specific topic labels, descriptions, or coherence flags
  from the list above.
- Do not introduce concepts, themes, or absences that cannot be traced to an actual topic.
- If you cannot name the specific topic, do not make the finding.
- Prefer fine-grained distinctions over broad summaries.
- Skip anything predictable given the group name.
- Skip the free/reduced lunch poverty framing — it appears everywhere and is already known.
- Skip generic engagement/motivation language unless the topic evidence makes it unusually specific.
- If fewer than two strong findings exist for a section, write one or write none. Do not pad.

Focus on:
1. SUBTHEME DISTINCTIONS — distinct needs, populations, pedagogies, use cases, or framing
   modes within this group.

2. INTERNAL FRAMING DIFFERENCES — cases within this group where similar underlying needs are
   presented through meaningfully different rhetorical frames, populations, or classroom
   contexts.

3. NOTABLE ABSENCES — a topic you would expect given the internal logic of this group's other
   topics, but that is missing.

4. REQUEST TRIGGERS AND PITCH STRATEGIES — patterns in why teachers say they are requesting
   within this group and how they frame that reason for a donor audience.

Formatting: use plain numbered section headers (e.g. '1. SUBTHEME DISTINCTIONS').
Do not use markdown headers (##). Do not use bullet points.

Format as four labeled sections. Under each, write 3-5 findings as short paragraphs.
Be specific — name the topic label every time. Do not use bullet points.
""".strip()

def build_per_group_prompt(group, topic_lines_text):
    return f"""
Below is a list of NMF topics discovered from teacher project request essays on DonorsChoose
for a single group: {group} ({GROUP_DESCRIPTION}).
Each topic represents a cluster of essays with similar language, framing, and request patterns.
{topic_lines_text}

{PER_GROUP_INSTRUCTIONS}
""".strip()

def _call_with_retry(prompt, max_retries=3):
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_SYNTHESIS,
                messages=[
                    {"role": "system", "content": SYNTHESIS_SYSTEM},
                    {"role": "user",   "content": prompt},
                ],
            )
            return resp.choices[0].message.content.strip()
        except (_openai.RateLimitError, _openai.APITimeoutError) as e:
            if attempt < max_retries:
                _time.sleep(2 ** attempt)
            else:
                raise
        except Exception as e:
            print(f"Non-retryable error: {e}")
            return None

synthesis_cross = _call_with_retry(SYNTHESIS_PROMPT_CROSS_GROUP)
if synthesis_cross is None:
    append_warning(
        WARNINGS_PATH,
        "03_insights_generation",
        "SYNTHESIS_CROSS_GROUP_FAILED",
        "Cross-group synthesis failed",
        context={},
    )
    raise RuntimeError("Cross-group synthesis failed; stopping before downstream cells.")

with open(OUT("analysis", "llm_synthesis_cross_group.txt"), "w", encoding="utf-8") as f:
    f.write(synthesis_cross)

groups = [
    g for g in labels_df[GROUPBY_FIELD].dropna().unique().tolist()
    if g not in EXCLUDE_GROUPS
]

def synthesize_one_group(group):
    topic_lines_text = build_topic_lines(labels_df, GROUPBY_FIELD, group=group)
    prompt = build_per_group_prompt(group, topic_lines_text)
    result = _call_with_retry(prompt)
    if result is None:
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "SYNTHESIS_GROUP_FAILED",
            f"Synthesis failed for group '{group}'",
            context={"group": group},
        )
        return group, None
    slug = slugify_group_value(group)
    fpath = OUT("analysis", f"llm_synthesis_{slug}.txt")
    with open(fpath, "w", encoding="utf-8") as f:
        f.write(result)
    return group, result

per_group_results = {}
SYNTHESIS_FAILED_GROUPS = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(synthesize_one_group, g): g for g in groups}
    for future in as_completed(futures):
        submitted_group = futures[future]
        try:
            group, result = future.result()
        except Exception as e:
            SYNTHESIS_FAILED_GROUPS.append(str(submitted_group))
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "SYNTHESIS_GROUP_FAILED",
                f"Synthesis failed for group '{submitted_group}'",
                context={"group": submitted_group, "error": str(e)},
            )
            print(f"FAILED: {submitted_group} | error: {e}")
            continue

        if result is not None:
            per_group_results[group] = result
            print(f"Done: {group}")
        else:
            SYNTHESIS_FAILED_GROUPS.append(str(group))
            print(f"FAILED: {group}")

per_group_results = {g: per_group_results[g] for g in groups if g in per_group_results}

if not per_group_results:
    raise RuntimeError("No per-group synthesis results were produced; stopping before downstream cells.")

Done: Las Vegas, NV
Done: Miami, FL
Done: Orlando, FL
Done: Tampa, FL
Done: Broward County, FL
Done: Los Angeles, CA
Done: San Diego, CA
Done: Houston, TX
Done: orlando_geo_group
Done: tampa_geo_group
Done: miami_geo_group


In [11]:
# ── EXTERNAL-FACING INSIGHTS — structured JSON call ───────────────────────
OUTPUT_GROUP_KEY = "by_group"
REQUIRED_GROUP_VALUES = list(per_group_results.keys())

if not REQUIRED_GROUP_VALUES:
    raise RuntimeError("No synthesized group results are available; cannot build external-facing insights.")

def strip_json_fences(text):
    text = (text or "").strip()
    text = re.sub(r"^\s*```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    return text.strip()

def normalize_source_topics(source_topics):
    """Normalize model output to [{'group': ..., 'topic_id': int}] with light recovery."""
    out = []

    for src in source_topics or []:
        group = None
        tid = None

        if isinstance(src, dict):
            group = src.get("group", src.get("group_value", ""))
            tid = src.get("topic_id", src.get("topic", src.get("id", "")))

        elif isinstance(src, str):
            s = src.strip()

            # Expected compact form: "<group>|<topic_id>"
            if "|" in s:
                left, right = s.rsplit("|", 1)
                group = left.strip()
                tid = right.strip()

            # Recovery form: "Topic 7"
            elif re.fullmatch(r"(?i)topic\s+\d+", s):
                tid = re.sub(r"(?i)topic\s+", "", s).strip()

        if group is not None:
            group = str(group).strip()
        if tid is not None:
            tid = str(tid).strip()

        if not tid:
            continue

        try:
            tid = int(float(tid))
        except Exception:
            continue

        # Only keep rows with an explicit group; skip ambiguous recovered topic-only rows
        if not group:
            continue

        out.append({"group": group, "topic_id": tid})

    deduped = []
    seen = set()
    for item in out:
        key = (item["group"], item["topic_id"])
        if key not in seen:
            seen.add(key)
            deduped.append(item)

    return deduped

def normalize_insight(insight):
    if not isinstance(insight, dict):
        raise ValueError(f"Insight must be a dict, got {type(insight)}")

    raw_source_topics = deepcopy(insight.get("source_topics", []))
    source_topics = normalize_source_topics(raw_source_topics)

    normalized = {
        "title": str(insight.get("title", "")).strip(),
        "what_seeing": str(insight.get("what_seeing", "")).strip(),
        "why_easy_to_miss": str(insight.get("why_easy_to_miss", "")).strip(),
        "what_it_means": str(insight.get("what_it_means", "")).strip(),
        "source_topics": source_topics,
        "source_topics_claimed": raw_source_topics,
    }
    if not normalized["title"]:
        raise ValueError(f"Insight missing title: {insight}")
    return normalized

synthesis_parts = []
if synthesis_cross:
    synthesis_parts.append(f"=== CROSS-GROUP ANALYSIS ===\n{synthesis_cross}")
for group_value, text in per_group_results.items():
    if text:
        synthesis_parts.append(f"=== {group_value} ===\n{text}")
synthesis_input = "\n\n".join(synthesis_parts)

INSIGHTS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You write precise, decision-facing briefings for external audiences: "
    "foundations, corporate partners, policymakers, and major donors. "
    "You return ONLY valid JSON. No preamble, no explanation, no markdown fences."
)

example_source_topics = []
example_rows = (
    labels_df[[GROUPBY_FIELD, "topic_id"]]
    .drop_duplicates()
    .head(3)
    .to_dict("records")
)

for row in example_rows:
    example_source_topics.append(f"{row[GROUPBY_FIELD]}|{int(row['topic_id'])}")

if not example_source_topics:
    for i, gv in enumerate(REQUIRED_GROUP_VALUES[:2], start=1):
        example_source_topics.append(f"{gv}|{i}")

example_source_topics_json = json.dumps(example_source_topics, ensure_ascii=False)

INSIGHTS_PROMPT = f"""
You are given a comprehensive structured analysis of classroom project request patterns including:
within-group subthemes, cross-group signatures, framing differences, notable absences,
and request triggers and pitch strategies.

The analysis is grouped by the field "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

{synthesis_input}

Your task: produce a decision-facing insights document for external stakeholders —
executives, foundation leaders, state legislators, and major donors — who are
intelligent but not close to classroom realities. These readers should finish each
insight thinking: "I never would have known that, and the fact that this organization
can see it means I should take their priorities seriously."

SELECTION CRITERIA:
- Each insight must surface something genuinely non-obvious about what classrooms need,
  reframe how a thoughtful outsider should interpret teacher demand, and have a clear
  implication for where attention or resources should go.
- Prioritize insights that reveal hidden scale, misallocated attention, or a shift in
  how teachers are operating that external stakeholders would not expect.
- Avoid methodological commentary, insights that only matter to data practitioners,
  and observations that merely restate what the group label already suggests.

COUNT + COVERAGE REQUIREMENTS:
- Return between 10 and 20 cross-group insights in "key_insights". Produce as many as
  the evidence strongly supports — do not pad to reach a target and do not stop early
  if strong findings remain.
- "{OUTPUT_GROUP_KEY}" must include every group value in this exact list:
  {json.dumps(REQUIRED_GROUP_VALUES, ensure_ascii=False)}
- Return 3-6 strongest defensible insights for each group value above.
- Do not omit group values silently.
- Prefer slightly weaker but still defensible coverage over omission.

INSIGHT STRUCTURE — for every insight use exactly these fields:
- title: a crisp, declarative headline that makes the finding feel like a reveal,
  not a description.
- what_seeing: 2-4 sentences describing the observed pattern in plain, concrete language.
  Write about what teachers are actually doing or asking for — not about the analysis.
- why_easy_to_miss: 1-2 sentences explaining why this pattern is invisible to someone
  looking at surface-level groups or conventional reporting.
- what_it_means: 2-4 sentences translating the pattern into consequence.
  This should land like a strategic implication, not an analytical note.
- source_topics: a list of the strongest grounding topics for the insight.
  Use only topics that directly support the claim.
  Usually 2-4 topics is enough, but do not force a fixed count.
  Prefer fewer well-matched topics over padding with weak ones.
  Required format:
    - each item must be a string
    - each string must be exactly "<group_value>|<topic_id>"
    - group_value must exactly match one of the group values shown in the analysis
    - topic_id must be the integer topic number for that group
  Do not use labels, category names, prose, or invented group names in place of group_value.
  Example for this run: {example_source_topics_json}
- Do not include a "section" field inside each insight.
  Section is implied by where the insight appears in the JSON.

STYLE:
- Direct, confident, and human — not academic, not corporate.
- No mention of "topics", "model", "NMF", "pipeline", "cluster", or analytical machinery.
- No hedging phrases like "may suggest" or "could indicate".
- No filler.
- Do not repeat the same implication across insights.

VALIDITY RULES:
- Return valid JSON only.
- The response is incomplete if any required group value is missing from "{OUTPUT_GROUP_KEY}".
- Do not return markdown fences or explanatory text.

Return a JSON object with exactly this structure:
{{
  "key_insights": [/* 10-20 insight objects */],
  "{OUTPUT_GROUP_KEY}": {{
    "<group value>": [/* 3-6 insight objects */],
    ...
  }}
}}
""".strip()

resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": INSIGHTS_SYSTEM},
        {"role": "user",   "content": INSIGHTS_PROMPT},
    ],
    response_format={"type": "json_object"},
)

raw_json = strip_json_fences(resp.choices[0].message.content)
insights_data = json.loads(raw_json)

if not isinstance(insights_data, dict):
    raise ValueError("Top-level insights response is not a JSON object.")

raw_key = insights_data.get("key_insights", [])
raw_by_group = insights_data.get(OUTPUT_GROUP_KEY, {})

if not isinstance(raw_key, list):
    raise ValueError("key_insights is not a list.")
if not isinstance(raw_by_group, dict):
    raise ValueError(f"{OUTPUT_GROUP_KEY} is not an object.")

normalized = {
    "key_insights": [normalize_insight(i) for i in raw_key],
    OUTPUT_GROUP_KEY: {},
}

missing_group_values = [g for g in REQUIRED_GROUP_VALUES if g not in raw_by_group]
if missing_group_values:
    raise ValueError(f"Model omitted required group values: {missing_group_values}")

extra_group_values = [g for g in raw_by_group.keys() if g not in REQUIRED_GROUP_VALUES]
if extra_group_values:
    print(f"WARNING: extra group values returned and ignored: {extra_group_values}")

for group_value in REQUIRED_GROUP_VALUES:
    items = raw_by_group.get(group_value, [])
    if not isinstance(items, list):
        raise ValueError(f"{OUTPUT_GROUP_KEY}['{group_value}'] is not a list.")
    normalized[OUTPUT_GROUP_KEY][group_value] = [normalize_insight(i) for i in items]

insights_data = normalized

if not (10 <= len(insights_data["key_insights"]) <= 20):
    raise ValueError(f"Expected between 10 and 20 key insights, got {len(insights_data['key_insights'])}")

bad_group_counts = {
    group_value: len(items)
    for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    if not (3 <= len(items) <= 6)
}
if bad_group_counts:
    raise ValueError(f"Expected between 3 and 6 insights per group value, got: {bad_group_counts}")

# Mid-cell write required by spec: post-normalize_insight(), pre-verification
write_json(OUT("insights", "insights_candidates.json"), insights_data)

PosixPath('/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_124534_miami_geo_group_45fd548b/insights/insights_candidates.json')

In [12]:
total_source_topics = (
    sum(len(i.get("source_topics", [])) for i in insights_data["key_insights"]) +
    sum(len(i.get("source_topics", [])) for items in insights_data[OUTPUT_GROUP_KEY].values() for i in items)
)

print(f"Key insights:  {len(insights_data.get('key_insights', []))}")
print(f"Group values:  {list(insights_data.get(OUTPUT_GROUP_KEY, {}).keys())}")
print(f"Total source_topics across all insights: {total_source_topics}")

Key insights:  12
Group values:  ['Broward County, FL', 'Houston, TX', 'Las Vegas, NV', 'Los Angeles, CA', 'Miami, FL', 'Orlando, FL', 'San Diego, CA', 'Tampa, FL', 'miami_geo_group', 'orlando_geo_group', 'tampa_geo_group']
Total source_topics across all insights: 159


In [13]:
VERIFY_SYSTEM = (
    "You are checking whether topic labels correctly support a given insight. "
    "Be strict. If the topic is only loosely related, drop it. "
    "Return only valid JSON."
)

def verify_source_topics(insight, labels_df, groupby_field, max_retries=3):
    title = insight.get("title", "")
    what_seeing = insight.get("what_seeing", "")
    what_it_means = insight.get("what_it_means", "")
    source_topics = insight.get("source_topics", [])
    warning_identifier = title[:120] if str(title).strip() else str(what_seeing).strip()[:120]

    if not source_topics:
        return insight

    topic_lines = []
    for src in source_topics:
        if isinstance(src, dict):
            group = src.get("group", "")
            tid = str(src.get("topic_id", ""))
        elif isinstance(src, str) and "|" in src:
            group, tid = src.rsplit("|", 1)
        else:
            continue

        match = labels_df[
            (labels_df[groupby_field] == group) &
            (labels_df["topic_id"] == int(float(tid)))
        ]
        if not match.empty:
            row = match.iloc[0]
            topic_lines.append(
                f"  {group}|{tid} | label: {row['proposed_label']} | "
                f"description: {row['description']}"
            )

    if not topic_lines:
        return insight

    prompt = f"""
Insight title: {title}

What we're seeing: {what_seeing}

What it means: {what_it_means}

Claimed source topics:
{chr(10).join(topic_lines)}

Return JSON: {{"verified_topics": [{{"group": "...", "topic_id": <int>}}, ...]}}
""".strip()

    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_SYNTHESIS,
                messages=[
                    {"role": "system", "content": VERIFY_SYSTEM},
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "json_object"},
            )
            result = json.loads(resp.choices[0].message.content)
            original_n = len(source_topics)
            insight["source_topics"] = normalize_source_topics(result.get("verified_topics", source_topics))
            verified_n = len(insight.get("source_topics", []))
            if verified_n != original_n:
                print(f"Adjusted source_topics: {title[:80]} | {original_n} -> {verified_n}")
            if verified_n == 0 and original_n > 0:
                print(f"WARNING: all source_topics removed: {title[:80]}")
            return insight
        except (_openai.RateLimitError, _openai.APITimeoutError) as e:
            if attempt < max_retries:
                _time.sleep(2 ** attempt)
            else:
                append_warning(
                    WARNINGS_PATH,
                    "03_insights_generation",
                    "VERIFY_API_FAILURE",
                    f"Verification failed for '{warning_identifier or '[untitled]'}'",
                    context={
                        "title": title or None,
                        "what_seeing_preview": str(what_seeing).strip()[:120] or None,
                        "error": str(e),
                    },
                )
                return insight
        except Exception as e:
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "VERIFY_API_FAILURE",
                f"Verification failed for '{warning_identifier or '[untitled]'}'",
                context={
                    "title": title or None,
                    "what_seeing_preview": str(what_seeing).strip()[:120] or None,
                    "error": str(e),
                },
            )
            return insight

for key in ["key_insights"]:
    insights_data[key] = [verify_source_topics(i, labels_df, GROUPBY_FIELD) for i in insights_data.get(key, [])]

for cat in insights_data.get(OUTPUT_GROUP_KEY, {}):
    insights_data[OUTPUT_GROUP_KEY][cat] = [
        verify_source_topics(i, labels_df, GROUPBY_FIELD)
        for i in insights_data[OUTPUT_GROUP_KEY][cat]
    ]

print("Verification complete.")

Adjusted source_topics: Teachers are not asking for “literacy” — they are solving three different readin | 6 -> 5
Adjusted source_topics: Flexible seating is functioning as classroom infrastructure, not décor. | 5 -> 3
Adjusted source_topics: Student wellness demand is splitting into two different markets: regulation spac | 5 -> 4
Adjusted source_topics: Devices are now basic access infrastructure, not enrichment. | 5 -> 2
Adjusted source_topics: Teachers are translating broad hardship into donor-legible bottlenecks. | 5 -> 4
Adjusted source_topics: Many ordinary supplies are now being pitched as systems that keep classrooms ope | 5 -> 4
Adjusted source_topics: Some of the strongest requests are not about academics at all — they are about g | 5 -> 4
Adjusted source_topics: Special education requests in Broward are not one category — they split by testi | 3 -> 2
Adjusted source_topics: Broward teachers are unusually skilled at turning broad need into named daily ro | 4 -> 3
Adjusted sou

In [14]:
# ── EVIDENCE PACK + CONFIDENCE SCORING ─────────────────────────────────────
assert "project_topic_bridge_df" in dir() and not project_topic_bridge_df.empty, (
    "project_topic_bridge_df missing — ensure the bridge-build cell ran successfully"
)

BRIDGE_LOOKUP = {
    topic_key: grp[["project_id", "weight", "topic_share"]].copy()
    for topic_key, grp in project_topic_bridge_df.groupby("topic_key", observed=True)
}

def get_topic_key(group, topic_id):
    return f"{GROUPBY_FIELD}={group}|topic={int(float(topic_id))}"

def get_project_ids(source_topics, bridge_lookup, max_ids=None):
    if not source_topics:
        return [], {}
    topic_rows_cache = {}
    records = []
    for src in source_topics:
        if isinstance(src, dict):
            group = str(src.get("group", "")).strip()
            topic_id = int(float(src.get("topic_id", -1)))
        elif isinstance(src, str) and "|" in src:
            group, topic_id = src.rsplit("|", 1)
            group = str(group).strip()
            topic_id = int(float(topic_id))
        else:
            continue
        topic_key = get_topic_key(group, topic_id)
        if topic_key not in topic_rows_cache:
            topic_rows_cache[topic_key] = bridge_lookup.get(
                topic_key,
                pd.DataFrame(columns=["project_id", "weight", "topic_share"]),
            )
        records.extend(topic_rows_cache[topic_key]["project_id"].tolist())
    seen = list(dict.fromkeys(records))
    return (seen if max_ids is None else seen[:max_ids]), topic_rows_cache

def iter_candidate_insights(data):
    idx = 1
    for insight in data.get("key_insights", []):
        yield {
            "insight_id": f"KI_{idx:03d}",
            "section": "key_insights",
            "category_bucket": None,
            "insight": insight,
        }
        idx += 1

    group_idx = 1
    for group_value, items in data.get(OUTPUT_GROUP_KEY, {}).items():
        for insight in items:
            yield {
                "insight_id": f"BG_{group_idx:03d}",
                "section": OUTPUT_GROUP_KEY,
                "category_bucket": group_value,
                "insight": insight,
            }
            group_idx += 1

candidate_rows = list(iter_candidate_insights(insights_data))

topic_rows_cache_global = {}
all_needed_pids = []
for row in candidate_rows:
    _, topic_rows_cache = get_project_ids(row["insight"].get("source_topics", []), BRIDGE_LOOKUP, max_ids=None)
    for topic_key, topic_rows in topic_rows_cache.items():
        if topic_key not in topic_rows_cache_global:
            topic_rows_cache_global[topic_key] = topic_rows
        all_needed_pids.extend(topic_rows["project_id"].tolist())

all_needed_pids = list(dict.fromkeys(all_needed_pids))

ESSAY_LOOKUP_PATH = ROOT / "OUTPUTS/prepared/02_project_essay_lookup.parquet"
if not ESSAY_LOOKUP_PATH.exists():
    raise FileNotFoundError(
        f"Essay lookup parquet not found: {ESSAY_LOOKUP_PATH}. Run Notebook 01 first."
    )

essay_lookup_df = pd.read_parquet(ESSAY_LOOKUP_PATH, columns=["project_id", "essay_text"])
essay_lookup = (
    essay_lookup_df[essay_lookup_df["project_id"].isin(all_needed_pids)]
    .drop_duplicates(subset=["project_id"], keep="first")
    .set_index("project_id")["essay_text"]
    .to_dict()
)

token_lookup = df.set_index("project_id")["tokens"].to_dict()

def _parse_topic_id(val):
    """Handle both plain ints and 'Topic N' strings from model output."""
    s = str(val).strip()
    if s.lower().startswith("topic"):
        s = s.split()[-1]
    return int(s)

label_index = {}
for _, row in labels_df.iterrows():
    try:
        tid = _parse_topic_id(row["topic_id"])
        label_index[(str(row[GROUPBY_FIELD]), tid)] = row
    except (ValueError, IndexError):
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "LABEL_INDEX_PARSE_FAILURE",
            f"Could not parse topic_id '{row['topic_id']}' for group '{row[GROUPBY_FIELD]}'",
            context={"group": str(row[GROUPBY_FIELD]), "topic_id": str(row["topic_id"])},
        )

def build_full_essay_text(project_id):
    if project_id in essay_lookup and str(essay_lookup[project_id]).strip():
        return re.sub(r"\s+", " ", str(essay_lookup[project_id])).strip()
    return ""

def build_injected_tokens(project_id):
    tokens = token_lookup.get(project_id, [])
    injected = [
        str(t).strip()
        for t in tokens
        if re.fullmatch(r"__[^_]+(?:_[^_]+)*__", str(t).strip())
    ]
    injected = list(dict.fromkeys(injected))
    return " | ".join(injected)

def build_evidence_obj(candidate):
    insight = candidate["insight"]
    source_topics = insight.get("source_topics", [])
    claimed_topic_count = len(insight.get("source_topics_claimed", []))
    supporting_project_ids, topic_rows_cache = get_project_ids(source_topics, topic_rows_cache_global, max_ids=None)

    topic_support = []
    for src in source_topics:
        if not isinstance(src, dict):
            continue
        group = str(src.get("group", ""))
        topic_id = int(float(src.get("topic_id", -1)))
        topic_key = get_topic_key(group, topic_id)
        topic_rows = topic_rows_cache.get(topic_key, pd.DataFrame(columns=["project_id", "weight", "topic_share"]))
        label_row = label_index.get((group, topic_id))
        topic_support.append({
            "group": group,
            "topic_id": topic_id,
            "topic_label": label_row["proposed_label"] if label_row is not None else "[not found in labels_df]",
            "topic_description": label_row["description"] if label_row is not None else "[not found in labels_df]",
            "coherence_flag": label_row["coherence_flag"] if label_row is not None else "unknown",
            "supporting_project_count": int(topic_rows["project_id"].nunique()) if not topic_rows.empty else 0,
            "mean_topic_share": float(topic_rows["topic_share"].mean()) if not topic_rows.empty else 0.0,
            "median_topic_share": float(topic_rows["topic_share"].median()) if not topic_rows.empty else 0.0,
        })

    sample_projects = []
    for pid in supporting_project_ids[:N_REPRESENTATIVE]:
        matches = []
        for src in source_topics:
            if not isinstance(src, dict):
                continue
            topic_key = get_topic_key(src.get("group", ""), src.get("topic_id", -1))
            topic_rows = topic_rows_cache.get(topic_key, pd.DataFrame(columns=["project_id", "topic_share"]))
            hit = topic_rows[topic_rows["project_id"] == pid]
            if not hit.empty:
                matches.append((src["group"], int(float(src["topic_id"])), float(hit["topic_share"].iloc[0])))
        if matches:
            best = sorted(matches, key=lambda x: x[2], reverse=True)[0]
            sample_projects.append({
                "project_id": pid,
                "group": best[0],
                "topic_id": best[1],
                "topic_share": best[2],
            })

    return {
        "schema_version": "v1",
        "run_id": RUN_ID,
        "insight_id": candidate["insight_id"],
        "section": candidate["section"],
        "category_bucket": candidate["category_bucket"],
        "group_by_field": GROUPBY_FIELD,
        "title": insight.get("title", ""),
        "what_seeing": insight.get("what_seeing", ""),
        "why_easy_to_miss": insight.get("why_easy_to_miss", ""),
        "what_it_means": insight.get("what_it_means", ""),
        "source_topics_claimed": insight.get("source_topics_claimed", []),
        "source_topics_verified": source_topics,
        "verification_drop_count": int(max(claimed_topic_count - len(source_topics), 0)),
        "supporting_project_count": int(len(supporting_project_ids)),
        "topic_support": topic_support,
        "sample_projects": sample_projects,
        "claimed_topic_count": int(claimed_topic_count),
    }

candidate_evidence_rows = [build_evidence_obj(row) for row in candidate_rows]

CONFIDENCE_CFG = dict(CFG["analysis"]["confidence"])

support_volume_scale_meta = get_effective_support_volume_scale(
    df=df,
    groupby_field=GROUPBY_FIELD,
    confidence_cfg=CONFIDENCE_CFG,
)

CONFIDENCE_CFG["support_volume_scale_effective"] = support_volume_scale_meta["effective_scale"]

print(
    f"Effective support_volume_scale_effective: {CONFIDENCE_CFG['support_volume_scale_effective']} "
    f"(multiplier={support_volume_scale_meta['scale_multiplier']}, "
    f"median_group_size={support_volume_scale_meta['median_group_size']:.1f})"
)
TOPIC_QUALITY_MAP = {"coherent": 1.0, "mixed": 0.6, "redundant": 0.2, "unclear": 0.0, "unknown": 0.0}

CONFIDENCE_RATIONALE_SYSTEM = (
    "You explain confidence scores for analytical insights. Return only one short sentence."
)

def build_confidence_rationale(conf_row):
    prompt = (
        f"Insight: {conf_row['title']}\n"
        f"Support volume score: {conf_row['support_volume_score']:.1f}\n"
        f"Support breadth score: {conf_row['support_breadth_score']:.1f}\n"
        f"Topic quality score: {conf_row['topic_quality_score']:.1f}\n"
        f"Bridge strength score: {conf_row['bridge_strength_score']:.1f}\n"
        f"Verification cleanliness score: {conf_row['verification_cleanliness_score']:.1f}\n"
        f"Final confidence: {conf_row['confidence_score']:.1f} ({conf_row['confidence_band']})\n"
        "Explain the main drivers in one short sentence."
    )
    try:
        resp = client.chat.completions.create(
            model=MODEL_LABELING,
            messages=[
                {"role": "system", "content": CONFIDENCE_RATIONALE_SYSTEM},
                {"role": "user", "content": prompt},
            ],
        )
        return resp.choices[0].message.content.strip()
    except Exception:
        return (
            f"Confidence is {conf_row['confidence_band']} because support volume, topic cleanliness, "
            f"and bridge strength combine to {conf_row['confidence_score']:.1f} after penalties."
        )

def score_insight_confidence(evidence_obj, cfg):
    verified_topic_coherence_flags = [ts["coherence_flag"] for ts in evidence_obj["topic_support"]]
    supporting_project_count = int(evidence_obj["supporting_project_count"])
    verified_topic_count = int(len(evidence_obj["source_topics_verified"]))
    claimed_topic_count = int(evidence_obj["claimed_topic_count"])

    support_volume_score = min(
        100.0,
        100.0 * supporting_project_count / cfg["support_volume_scale_effective"],
    )
    support_breadth_score = min(
        100.0,
        100.0 * verified_topic_count / cfg["support_breadth_scale_topics"],
    )
    topic_quality_score = (
        0.0 if not verified_topic_coherence_flags
        else 100.0 * mean(TOPIC_QUALITY_MAP.get(flag, 0.0) for flag in verified_topic_coherence_flags)
    )
    mean_topic_share_all_verified_topics = (
        0.0 if not evidence_obj["topic_support"]
        else float(mean(ts["mean_topic_share"] for ts in evidence_obj["topic_support"]))
    )
    bridge_strength_score = (
        0.0 if not evidence_obj["topic_support"]
        else min(100.0, 100.0 * mean_topic_share_all_verified_topics / cfg["bridge_strength_scale_topic_share"])
    )
    
    verification_ratio = verified_topic_count / claimed_topic_count
    verification_cleanliness_score = 100.0 * (verification_ratio ** 2)
    
    confidence_score = (
        0.30 * support_volume_score +
        0.10 * support_breadth_score +
        0.20 * topic_quality_score +
        0.25 * bridge_strength_score +
        0.15 * verification_cleanliness_score
    )
    confidence_score = round(max(0.0, min(100.0, confidence_score)), 1)

    if confidence_score >= cfg["band_thresholds"]["high"]:
        confidence_band = "high"
    elif confidence_score >= cfg["band_thresholds"]["medium"]:
        confidence_band = "medium"
    else:
        confidence_band = "low"

    # ── confidence_rationale is intentionally excluded here.
    # It is built separately in parallel below and merged back into conf_rows.
    return {
        "support_volume_score": round(support_volume_score, 1),
        "support_breadth_score": round(support_breadth_score, 1),
        "topic_quality_score": round(topic_quality_score, 1),
        "bridge_strength_score": round(bridge_strength_score, 1),
        "verification_cleanliness_score": round(verification_cleanliness_score, 1),
        "confidence_score": confidence_score,
        "confidence_band": confidence_band,
        "confidence_rationale": None,  # placeholder — filled by parallel block below
        "supporting_project_count": supporting_project_count,
        "verified_topic_count": verified_topic_count,
        "claimed_topic_count": claimed_topic_count,
        "mean_topic_share_all_verified_topics": round(mean_topic_share_all_verified_topics, 4),
        "verified_topic_coherence_flags": verified_topic_coherence_flags,
    }

conf_rows = []
for ev in candidate_evidence_rows:
    conf = score_insight_confidence(ev, CONFIDENCE_CFG)
    conf["insight_id"] = ev["insight_id"]
    conf["title"] = ev["title"]
    conf["confidence_version"] = "v1"
    conf_rows.append(conf)

# ── parallel rationale generation ─────────────────────────────────────────────
def _build_rationale_for_row(conf_row):
    rationale = build_confidence_rationale(conf_row)
    return conf_row["insight_id"], rationale

rationale_map = {}
with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
    futures = {executor.submit(_build_rationale_for_row, row): row["insight_id"]
               for row in conf_rows}
    for future in as_completed(futures):
        try:
            insight_id, rationale = future.result()
            rationale_map[insight_id] = rationale
        except Exception as e:
            insight_id = futures[future]
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "RATIONALE_API_FAILURE",
                f"Confidence rationale failed for '{insight_id}'",
                context={"insight_id": insight_id, "error": str(e)},
            )
            rationale_map[insight_id] = None

for row in conf_rows:
    row["confidence_rationale"] = rationale_map.get(row["insight_id"])

conf_df = pd.DataFrame(conf_rows)
conf_df.to_csv(OUT("insights", "insights_confidence.csv"), index=False)
print(f"Scored {len(conf_df):,} candidate insights")

# ── STRICT CHANGE VERIFICATION ────────────────────────────────────────────────
# Confirms that the only structural change from the original is the parallelization
# of build_confidence_rationale. All other logic, keys, values, and outputs must
# be identical. Raises AssertionError immediately on any violation.

_expected_conf_df_columns = {
    "support_volume_score", "support_breadth_score", "topic_quality_score",
    "bridge_strength_score", "verification_cleanliness_score",
    "confidence_score", "confidence_band", "confidence_rationale",
    "supporting_project_count", "verified_topic_count", "claimed_topic_count",
    "mean_topic_share_all_verified_topics", "verified_topic_coherence_flags",
    "insight_id", "title", "confidence_version",
}
_actual_cols = set(conf_df.columns)
_missing = _expected_conf_df_columns - _actual_cols
_extra   = _actual_cols - _expected_conf_df_columns
assert not _missing, f"CHANGE VERIFICATION FAILED — columns missing from conf_df: {_missing}"
assert not _extra,   f"CHANGE VERIFICATION FAILED — unexpected new columns in conf_df: {_extra}"

assert len(conf_df) == len(candidate_evidence_rows), (
    f"CHANGE VERIFICATION FAILED — conf_df row count {len(conf_df)} "
    f"does not match candidate_evidence_rows count {len(candidate_evidence_rows)}"
)

assert conf_df["confidence_version"].eq("v1").all(), (
    "CHANGE VERIFICATION FAILED — confidence_version is not 'v1' for all rows"
)

assert conf_df["confidence_rationale"].notna().all() or rationale_map, (
    "CHANGE VERIFICATION FAILED — confidence_rationale column is entirely null "
    "and rationale_map is empty; parallel block may not have executed"
)

_rationale_none_count = conf_df["confidence_rationale"].isna().sum()
if _rationale_none_count > 0:
    print(f"WARNING: {_rationale_none_count} insight(s) have null confidence_rationale "
          f"due to API failures — check warnings_03.jsonl for RATIONALE_API_FAILURE entries")

assert set(conf_df["insight_id"]) == {row["insight_id"] for row in candidate_evidence_rows}, (
    "CHANGE VERIFICATION FAILED — insight_id set in conf_df does not match candidate_evidence_rows"
)

_score_cols = [
    "support_volume_score", "support_breadth_score", "topic_quality_score",
    "bridge_strength_score", "verification_cleanliness_score",
]
for col in _score_cols:
    assert conf_df[col].between(0.0, 100.0).all(), (
        f"CHANGE VERIFICATION FAILED — {col} contains values outside [0, 100]"
    )

assert conf_df["confidence_score"].between(0.0, 100.0).all(), (
    "CHANGE VERIFICATION FAILED — confidence_score contains values outside [0, 100]"
)

assert conf_df["confidence_band"].isin({"high", "medium", "low"}).all(), (
    "CHANGE VERIFICATION FAILED — confidence_band contains unexpected values: "
    f"{conf_df['confidence_band'].unique().tolist()}"
)

_conf_output_path = OUT("insights", "insights_confidence.csv")
assert _conf_output_path.exists(), (
    f"CHANGE VERIFICATION FAILED — insights_confidence.csv was not written to {_conf_output_path}"
)

print("Change verification passed — conf_df structure, row count, score ranges, "
      "bands, insight_id alignment, and output file all confirmed.")

Effective support_volume_scale_effective: 4023 (multiplier=0.25, median_group_size=16091.5)
Scored 54 candidate insights
Change verification passed — conf_df structure, row count, score ranges, bands, insight_id alignment, and output file all confirmed.


---

In [15]:
# ── CONFIDENCE CALIBRATION DIAGNOSTIC ──────────────────────────────────────
# NEW CELL — insert after confidence scoring cell, before tiering cell.
# Read-only. No side effects. Purpose: surface whether formula scale params
# and tiering thresholds will discriminate meaningfully on this corpus.
# Adjust params.yaml [analysis][confidence] and [analysis][tiering] based
# on the output, then re-run from the confidence scoring cell if needed.

conf_cfg = dict(CONFIDENCE_CFG)
tier_cfg = CFG["analysis"]["tiering"]

support_volume_scale_meta = get_effective_support_volume_scale(
    df=df,
    groupby_field=GROUPBY_FIELD,
    confidence_cfg=conf_cfg,
)
current_support_volume_scale_multiplier = support_volume_scale_meta["scale_multiplier"]
current_support_volume_scale_effective = support_volume_scale_meta["effective_scale"]
current_median_group_size = support_volume_scale_meta["median_group_size"]

# ── Join supporting_project_count and verified_topic_count from evidence pack
# (these are inputs to the confidence formula, not stored in conf_df itself)
evidence_index = {
    row["insight_id"]: row
    for row in candidate_evidence_rows
}

diag = conf_df.copy()

diag["supporting_project_count"] = diag["insight_id"].map(
    lambda iid: evidence_index[iid]["supporting_project_count"]
    if iid in evidence_index else 0
)
diag["verified_topic_count"] = diag["insight_id"].map(
    lambda iid: len(evidence_index[iid].get("source_topics_verified", []))
    if iid in evidence_index else 0
)

# Collect all per-topic mean_topic_share values for bridge strength calibration
all_mean_shares = [
    ts["mean_topic_share"]
    for row in candidate_evidence_rows
    for ts in row.get("topic_support", [])
    if "mean_topic_share" in ts
]

component_cols = [
    "support_volume_score",
    "support_breadth_score",
    "topic_quality_score",
    "bridge_strength_score",
    "verification_cleanliness_score",
]

# ── 1. Header ──────────────────────────────────────────────────────────────
n = len(diag)
print("=" * 65)
print("  CONFIDENCE CALIBRATION DIAGNOSTIC")
print(f"  insights scored: {n}  |  groupby: {GROUPBY_FIELD}")
print("=" * 65)

# ── 2. Component distributions ─────────────────────────────────────────────
print("\n── Component score distributions (0–100 scale) ──")
print(
    diag[component_cols + ["confidence_score"]]
    .describe(percentiles=[.1, .25, .5, .75, .9])
    .round(1)
    .to_string()
)

# ── 3. Ceiling hits — components that aren't discriminating ────────────────
print("\n── Ceiling hits (score >= 99.9) — components maxed for too many insights ──")
any_ceiling_problem = False
for col in component_cols:
    maxed = (diag[col] >= 99.9).sum()
    pct   = 100 * maxed / n if n else 0
    flag  = "  ⚠  not discriminating" if pct > 70 else ""
    print(f"  {col:<42s}: {maxed:>3}/{n}  ({pct:5.1f}%){flag}")
    if pct > 70:
        any_ceiling_problem = True

if any_ceiling_problem:
    print("\n  Current scale params (from params.yaml):")
    print(f"    support_volume_scale_median_size  = {current_support_volume_scale_multiplier}")
    print(f"    effective_support_volume_scale    = {current_support_volume_scale_effective}")
    print(f"    support_breadth_scale_topics      = {conf_cfg['support_breadth_scale_topics']}")
    print(f"    bridge_strength_scale_topic_share = {conf_cfg['bridge_strength_scale_topic_share']}")
    print("\n  Suggested recalibration targets (p75 of this corpus):")
    p75_spc   = int(diag['supporting_project_count'].quantile(0.75))
    p75_vtc   = float(round(diag['verified_topic_count'].quantile(0.75), 1))
    p75_share = float(round(pd.Series(all_mean_shares).quantile(0.75), 3)) if all_mean_shares else 'n/a'
    suggested_scale_multiplier = (
        round(p75_spc / current_median_group_size, 3)
        if current_median_group_size > 0 else "n/a"
    )
    print(f"    support_volume_scale_median_size  → {suggested_scale_multiplier}")
    print(f"    support_breadth_scale_topics      → {p75_vtc}")
    print(f"    bridge_strength_scale_topic_share → {p75_share}")
    
# ── 4. Actual support count distribution — key for scale calibration ───────
print("\n── supporting_project_count distribution (raw corpus values) ──")
print(
    diag["supporting_project_count"]
    .describe(percentiles=[.1, .25, .5, .75, .9])
    .round(0)
    .to_string()
)
print(f"\n  support_volume_scale_median_size currently set to: {current_support_volume_scale_multiplier}")
print(f"  effective support-volume denominator: {current_support_volume_scale_effective}")
print(f"  (anything at or above this scores 100 on support_volume_score)")
frac_maxed = (diag["supporting_project_count"] >= current_support_volume_scale_effective).mean()
print(f"  fraction of insights at or above that threshold: {frac_maxed:.0%}")

# ── 5. Final confidence_score band breakdown ───────────────────────────────
print("\n── confidence_score band distribution ──")
for band in ["high", "medium", "low"]:
    cnt = (diag["confidence_band"] == band).sum()
    pct = 100 * cnt / n if n else 0
    print(f"  {band:<8s}: {cnt:>3}  ({pct:5.1f}%)")

# ── 6. Simulate tiering under current thresholds ──────────────────────────
print("\n── Simulated tier distribution (what the tiering cell will produce) ──")

def _simulate_tier(row):
    cs  = row["confidence_score"]
    spc = row["supporting_project_count"]
    vtc = row["verified_topic_count"]
    tqs = row["topic_quality_score"]
    bss = row["bridge_strength_score"]
    top = tier_cfg["top"]
    sec = tier_cfg["secondary"]
    ter = tier_cfg["tertiary"]
    if (cs  >= top["min_confidence_score"] and
        spc >= top["min_supporting_project_count"] and
        vtc >= top["min_verified_topic_count"] and
        tqs >= top["min_topic_quality_score"] and
        bss >= top["min_bridge_strength_score"]):
        return "top"
    if (cs  >= sec["min_confidence_score"] and
        spc >= sec["min_supporting_project_count"] and
        tqs >= sec["min_topic_quality_score"] and
        bss >= sec["min_bridge_strength_score"]):
        return "secondary"
    if (cs  >= ter["min_confidence_score"] and
        spc >= ter["min_supporting_project_count"] and
        tqs >= ter["min_topic_quality_score"] and
        bss >= ter["min_bridge_strength_score"]):
        return "tertiary"
    return "rejected"

diag["_sim_tier"] = diag.apply(_simulate_tier, axis=1)

for tier in ["top", "secondary", "tertiary", "rejected"]:
    cnt = (diag["_sim_tier"] == tier).sum()
    pct = 100 * cnt / n if n else 0
    print(f"  {tier:<12s}: {cnt:>3}  ({pct:5.1f}%)")

# ── 7. Dead zone: rejected but not obviously weak ─────────────────────────
dead_zone = diag[
    (diag["_sim_tier"] == "rejected") &
    (diag["confidence_score"]         >= 40) &
    (diag["supporting_project_count"] >= 10)
].copy()

if not dead_zone.empty:
    print(f"\n── Dead zone: {len(dead_zone)} insight(s) rejected despite moderate support ──")
    print("   These fall through tier gaps. Review tertiary thresholds.")
    show_cols = [
        "insight_id", "confidence_score", "confidence_band",
        "supporting_project_count", "verified_topic_count",
        "topic_quality_score", "bridge_strength_score",
    ]
    print(dead_zone[show_cols].to_string(index=False))
else:
    print("\n  No dead-zone insights. Tier thresholds look clean.")

print("\n" + "=" * 65)
print("  If ceiling hits > 70% or tier distribution looks off,")
print("  adjust params.yaml and re-run from the confidence scoring cell.")
print("=" * 65)

  CONFIDENCE CALIBRATION DIAGNOSTIC
  insights scored: 54  |  groupby: miami_geo_group

── Component score distributions (0–100 scale) ──
       support_volume_score  support_breadth_score  topic_quality_score  bridge_strength_score  verification_cleanliness_score  confidence_score
count                  54.0                   54.0                 54.0                   54.0                            54.0              54.0
mean                   42.9                   70.4                 95.7                   72.2                            77.3              68.7
std                    38.5                   29.4                 16.2                   36.8                            31.3              22.4
min                     0.0                    0.0                  0.0                    0.0                             0.0               0.0
10%                     0.0                   33.3                100.0                    0.0                            28.3           

---

In [16]:
# ── TIERING + FINAL OUTPUTS + REPORTING ─────────────────────────────────────
TIER_CFG = CFG["analysis"]["tiering"]

def assign_insight_tier(confidence_obj, cfg):
    supporting_project_count = confidence_obj["supporting_project_count"]
    verified_topic_count = confidence_obj["verified_topic_count"]
    claimed_topic_count = confidence_obj["claimed_topic_count"]
    confidence_score = confidence_obj["confidence_score"]
    topic_quality_score = confidence_obj["topic_quality_score"]
    bridge_strength_score = confidence_obj["bridge_strength_score"]
    early_rejected_reason_code = confidence_obj.get("early_rejected_reason_code")

    if early_rejected_reason_code:
        return "rejected", early_rejected_reason_code
    if claimed_topic_count > 0 and verified_topic_count == 0:
        return "rejected", "verification_failed"

    top = cfg["top"]
    sec = cfg["secondary"]
    ter = cfg["tertiary"]

    if (
        confidence_score >= top["min_confidence_score"] and
        supporting_project_count >= top["min_supporting_project_count"] and
        verified_topic_count >= top["min_verified_topic_count"] and
        topic_quality_score >= top["min_topic_quality_score"] and
        bridge_strength_score >= top["min_bridge_strength_score"]
    ):
        return "top", None
    
    if (
        confidence_score >= sec["min_confidence_score"] and
        supporting_project_count >= sec["min_supporting_project_count"] and
        topic_quality_score >= sec["min_topic_quality_score"] and
        bridge_strength_score >= sec["min_bridge_strength_score"]
    ):
        return "secondary", None

    if (
        confidence_score >= ter["min_confidence_score"] and
        supporting_project_count >= ter["min_supporting_project_count"] and
        topic_quality_score >= ter["min_topic_quality_score"] and
        bridge_strength_score >= ter["min_bridge_strength_score"]
    ):
        return "tertiary", None

    if supporting_project_count < ter["min_supporting_project_count"]:
        return "rejected", "low_support"
    if topic_quality_score < ter["min_topic_quality_score"]:
        return "rejected", "low_topic_quality"
    if bridge_strength_score < ter["min_bridge_strength_score"]:
        return "rejected", "low_bridge_strength"
    return "rejected", "below_thresholds"

tier_rows = []
for _, row in conf_df.iterrows():
    tier, rejected_reason_code = assign_insight_tier(row.to_dict(), TIER_CFG)
    tier_rows.append({
        "insight_id": row["insight_id"],
        "tier": tier,
        "rejected_reason_code": rejected_reason_code,
        "tier_version": "v1",
    })
tier_df = pd.DataFrame(tier_rows)

run_warnings_count = get_warning_count(WARNINGS_PATH)

flat_rows = []
for ev in candidate_evidence_rows:
    conf = conf_df[conf_df["insight_id"] == ev["insight_id"]].iloc[0].to_dict()
    tier = tier_df[tier_df["insight_id"] == ev["insight_id"]].iloc[0].to_dict()
    flat_rows.append({
        "run_id": RUN_ID,
        "insight_id": ev["insight_id"],
        "section": ev["section"],
        "category_bucket": ev["category_bucket"],
        "title": ev["title"],
        "what_seeing": ev["what_seeing"],
        "why_easy_to_miss": ev["why_easy_to_miss"],
        "what_it_means": ev["what_it_means"],
        "confidence_score": conf["confidence_score"],
        "confidence_band": conf["confidence_band"],
        "confidence_version": conf["confidence_version"],
        "confidence_rationale": conf["confidence_rationale"],
        "tier": tier["tier"],
        "tier_version": tier["tier_version"],
        "rejected_reason_code": tier["rejected_reason_code"],
        "supporting_project_count": conf["supporting_project_count"],
        "verified_topic_count": conf["verified_topic_count"],
        "source_topic_count_claimed": conf["claimed_topic_count"],
        "source_topic_count_verified": conf["verified_topic_count"],
        "group_by_field": GROUPBY_FIELD,
        "filter_logic": FILTER_LOGIC,
        "warnings_count": run_warnings_count,
    })

insights_flat_df = pd.DataFrame(flat_rows)
insights_flat_df.to_csv(OUT("insights", "insights_flat.csv"), index=False)

accepted_ids = set(insights_flat_df[insights_flat_df["tier"] != "rejected"]["insight_id"])
rejected_ids = set(insights_flat_df[insights_flat_df["tier"] == "rejected"]["insight_id"])

accepted_evidence_rows = [row for row in candidate_evidence_rows if row["insight_id"] in accepted_ids]
rejected_evidence_rows = [row for row in candidate_evidence_rows if row["insight_id"] in rejected_ids]

write_json(OUT("insights", "rejected_insights.json"), [
    {
        **row,
        "confidence": conf_df[conf_df["insight_id"] == row["insight_id"]].iloc[0].to_dict(),
        "tiering": tier_df[tier_df["insight_id"] == row["insight_id"]].iloc[0].to_dict(),
    }
    for row in rejected_evidence_rows
])

structured = {"key_insights": [], OUTPUT_GROUP_KEY: {}}
for candidate in candidate_rows:
    ev = next((r for r in candidate_evidence_rows if r["insight_id"] == candidate["insight_id"]), None)
    if candidate["insight_id"] not in accepted_ids or ev is None:
        continue
    insight_clean = {
        "insight_id": candidate["insight_id"],
        "title": ev["title"],
        "what_seeing": ev["what_seeing"],
        "why_easy_to_miss": ev["why_easy_to_miss"],
        "what_it_means": ev["what_it_means"],
        "source_topics": ev["source_topics_verified"],
        "confidence_score": float(conf_df.loc[conf_df["insight_id"] == candidate["insight_id"], "confidence_score"].iloc[0]),
        "confidence_band": conf_df.loc[conf_df["insight_id"] == candidate["insight_id"], "confidence_band"].iloc[0],
        "confidence_version": "v1",
        "tier": tier_df.loc[tier_df["insight_id"] == candidate["insight_id"], "tier"].iloc[0],
        "tier_version": "v1",
        "supporting_project_count": int(ev["supporting_project_count"]),
        "verified_topic_count": int(len(ev["source_topics_verified"])),
        "warnings": [],
    }
    if candidate["section"] == "key_insights":
        structured["key_insights"].append(insight_clean)
    else:
        structured[OUTPUT_GROUP_KEY].setdefault(candidate["category_bucket"], []).append(insight_clean)

write_json(OUT("insights", "insights_structured.json"), structured)

# Accepted-only evidence artifacts
with open(OUT("evidence", "evidence_pack.jsonl"), "w", encoding="utf-8") as f:
    for row in accepted_evidence_rows:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

evidence_topic_support_rows = []
evidence_snippet_rows = []
evidence_project_rows = []
for row in accepted_evidence_rows:
    topic_rows_cache = {get_topic_key(ts["group"], ts["topic_id"]): BRIDGE_LOOKUP.get(get_topic_key(ts["group"], ts["topic_id"]), pd.DataFrame(columns=["project_id", "weight", "topic_share"])) for ts in row["topic_support"]}
    for ts in row["topic_support"]:
        evidence_topic_support_rows.append({
            "run_id": RUN_ID,
            "insight_id": row["insight_id"],
            "section": row["section"],
            "category_bucket": row["category_bucket"],
            "group_by_field": GROUPBY_FIELD,
            "group_value": ts["group"],
            "topic_id": ts["topic_id"],
            "topic_label": ts["topic_label"],
            "topic_description": ts["topic_description"],
            "coherence_flag": ts["coherence_flag"],
            "supporting_project_count": ts["supporting_project_count"],
            "mean_topic_share": ts["mean_topic_share"],
            "median_topic_share": ts["median_topic_share"],
        })
        topic_key = get_topic_key(ts["group"], ts["topic_id"])
        bridge_rows = topic_rows_cache.get(topic_key, pd.DataFrame(columns=["project_id", "weight", "topic_share"]))
        for _, br in bridge_rows.iterrows():
            evidence_project_rows.append({
                "run_id": RUN_ID,
                "insight_id": row["insight_id"],
                "group_by_field": GROUPBY_FIELD,
                "group_value": ts["group"],
                "topic_id": ts["topic_id"],
                "project_id": br["project_id"],
                "topic_key": topic_key,
                "weight": br["weight"],
                "topic_share": br["topic_share"],
            })
    for sp in row["sample_projects"]:
        evidence_snippet_rows.append({
            "run_id": RUN_ID,
            "insight_id": row["insight_id"],
            "group_by_field": GROUPBY_FIELD,
            "group_value": sp["group"],
            "topic_id": sp["topic_id"],
            "project_id": sp["project_id"],
            "topic_share": sp["topic_share"],
            "essay_text": build_full_essay_text(sp["project_id"]),
            "injected_tokens": build_injected_tokens(sp["project_id"]),
        })

pd.DataFrame(evidence_topic_support_rows).to_csv(OUT("evidence", "evidence_topic_support.csv"), index=False)
pd.DataFrame(evidence_snippet_rows).to_csv(OUT("evidence", "evidence_snippets.csv"), index=False)
pd.DataFrame(evidence_project_rows).to_csv(OUT("evidence", "evidence_projects.csv"), index=False)

# Chart-ready accepted-only data
chart_rows = []
group_support_rows = []
for row in accepted_evidence_rows:
    flat = insights_flat_df[insights_flat_df["insight_id"] == row["insight_id"]].iloc[0]
    chart_rows.append({
        "run_id": RUN_ID,
        "insight_id": row["insight_id"],
        "theme": row["title"],
        "section": row["section"],
        "category_bucket": row["category_bucket"],
        "tier": flat["tier"],
        "confidence_band": flat["confidence_band"],
        "confidence_score": flat["confidence_score"],
        "group_by_field": GROUPBY_FIELD,
        "supporting_project_count": row["supporting_project_count"],
        "verified_topic_count": len(row["source_topics_verified"]),
        "magnitude": row["supporting_project_count"],
        "magnitude_unit": "projects",
        "delta": None,
        "delta_unit": None,
        "period": None,
    })
    for ts in row["topic_support"]:
        group_support_rows.append({
            "run_id": RUN_ID,
            "insight_id": row["insight_id"],
            "theme": row["title"],
            "group_by_field": GROUPBY_FIELD,
            "group_value": ts["group"],
            "topic_id": ts["topic_id"],
            "supporting_project_count": ts["supporting_project_count"],
            "mean_topic_share": ts["mean_topic_share"],
            "confidence_score": flat["confidence_score"],
            "confidence_band": flat["confidence_band"],
            "tier": flat["tier"],
        })

pd.DataFrame(chart_rows).to_csv(OUT("chart_data", "chart_ready_insights.csv"), index=False)
pd.DataFrame(group_support_rows).to_csv(OUT("chart_data", "chart_ready_group_support.csv"), index=False)

# DOCX report accepted-only
def add_heading(doc, text, level):
    p = doc.add_heading(text, level=level)
    p.runs[0].font.color.rgb = RGBColor(0, 0, 0)
    return p

def add_insight(doc, insight):
    p = doc.add_paragraph()
    run = p.add_run(insight.get("title", ""))
    run.bold = True
    run.font.size = Pt(11)

    for label, key in [
        ("What we're seeing:", "what_seeing"),
        ("Why this is easy to miss:", "why_easy_to_miss"),
        ("What this means for funders:", "what_it_means"),
    ]:
        p = doc.add_paragraph()
        label_run = p.add_run(label + "  ")
        label_run.bold = True
        label_run.font.size = Pt(10)
        body_run = p.add_run(insight.get(key, ""))
        body_run.font.size = Pt(10)
        p.paragraph_format.space_after = Pt(2)
    doc.add_paragraph()

doc = Document()
for section in doc.sections:
    section.top_margin = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin = Inches(1)
    section.right_margin = Inches(1)

style = doc.styles["Normal"]
style.font.name = "Arial"
style.font.size = Pt(10)

add_heading(doc, f"Investigated Across {GROUPBY_FIELD}", level=1)
add_heading(doc, "Key Insights", level=2)
for insight in structured.get("key_insights", []):
    add_insight(doc, insight)

add_heading(doc, "By Group", level=1)
for category, items in structured.get(OUTPUT_GROUP_KEY, {}).items():
    add_heading(doc, category, level=2)
    for insight in items:
        add_insight(doc, insight)

docx_path = OUT("reports", "trend_tracker_report.docx")
doc.save(docx_path)

# Metadata / manifests
write_json(OUT("metadata", "confidence_model_meta.json"), {
    "schema_version": "v1",
    "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD,
    "filter_fields_key": FILTER_FIELDS_KEY,
    "confidence_config": CONFIDENCE_CFG,
})
write_json(OUT("metadata", "tiering_model_meta.json"), {
    "schema_version": "v1",
    "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD,
    "filter_fields_key": FILTER_FIELDS_KEY,
    "tiering_config": TIER_CFG,
})

groups_failed = sorted(set(NMF_GROUPS_FAILED) | set(LABELING_FAILED_GROUPS) | set(SYNTHESIS_FAILED_GROUPS))
eligible_groups = list(per_group_results.keys())

stage_manifest_path = OUT("metadata", "stage_manifest_03_insights_generation.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status="success",
    input_artifacts=[
        artifact_meta(ROOT / "OUTPUTS/prepared/05_enriched.parquet", "enriched_parquet"),
        artifact_meta(ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json", "stage_manifest_01"),
        artifact_meta(ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json", "stage_manifest_02"),
    ],
    output_artifacts=[
        artifact_meta(OUT("analysis", "category_tfidf.csv"), "category_tfidf_csv"),
        artifact_meta(OUT("analysis", "nmf_topics.csv"), "nmf_topics_csv"),
        artifact_meta(OUT("analysis", "nmf_weights.csv"), "nmf_weights_csv"),
        artifact_meta(OUT("analysis", "project_topic_bridge.csv"), "project_topic_bridge_csv"),
        artifact_meta(OUT("analysis", "llm_topic_labels.json"), "llm_topic_labels_json"),
        artifact_meta(OUT("insights", "insights_candidates.json"), "insights_candidates_json"),
        artifact_meta(OUT("insights", "insights_structured.json"), "insights_structured_json"),
        artifact_meta(OUT("insights", "rejected_insights.json"), "rejected_insights_json"),
        artifact_meta(OUT("insights", "insights_flat.csv"), "insights_flat_csv"),
        artifact_meta(OUT("insights", "insights_confidence.csv"), "insights_confidence_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_insights.csv"), "chart_ready_insights_csv"),
        artifact_meta(OUT("reports", "trend_tracker_report.docx"), "trend_tracker_report_docx"),
    ],
    row_counts={
        "input_projects": int(len(raw_df)),
        "filtered_projects": int(len(df)),
        "eligible_groups": int(len(eligible_groups)),
        "groups_skipped": int(len(NMF_GROUPS_SKIPPED)),
        "groups_failed": int(len(groups_failed)),
        "topics_generated": int(len(topics_df)),
        "insights_generated": int(len(candidate_evidence_rows)),
        "insights_accepted": int(len(accepted_ids)),
        "insights_rejected": int(len(rejected_ids)),
    },
    key_params={
        "group_by": GROUPBY_FIELD,
        "filter_fields_key": FILTER_FIELDS_KEY,
        "min_group_projects": CFG["analysis"]["min_group_projects"],
        "topic_assignment_threshold": CFG["analysis"]["topic_assignment_threshold"],
        "model_labeling": MODEL_LABELING,
        "model_synthesis": MODEL_SYNTHESIS,
        "confidence_config": CONFIDENCE_CFG,
        "tiering_config": TIER_CFG,
    },
    warnings_path=WARNINGS_PATH,
)

pipeline_manifest_path = OUT("metadata", "pipeline_manifest.json")
build_pipeline_manifest(
    output_path=pipeline_manifest_path,
    run_id=RUN_ID,
    run_date=RUN_DATE,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
    filter_spec_path=FILTER_SPEC_PATH,
    filter_summary_path=FILTER_SUMMARY_PATH,
    stage_manifest_paths=[
        ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json",
        ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json",
        stage_manifest_path,
    ],
    warnings_01_path=ROOT / "OUTPUTS/prepared/warnings_01.jsonl",
    warnings_02_path=ROOT / "OUTPUTS/enrichment/warnings_02.jsonl",
    warnings_03_path=WARNINGS_PATH,
    final_outputs={
        "insights_structured_json": str(OUT("insights", "insights_structured.json")),
        "insights_flat_csv": str(OUT("insights", "insights_flat.csv")),
        "evidence_pack_jsonl": str(OUT("evidence", "evidence_pack.jsonl")),
        "insights_confidence_csv": str(OUT("insights", "insights_confidence.csv")),
        "chart_ready_insights_csv": str(OUT("chart_data", "chart_ready_insights.csv")),
        "trend_tracker_report_docx": str(OUT("reports", "trend_tracker_report.docx")),
    },
)

print(f"Accepted insights: {len(accepted_ids):,}")
print(f"Rejected insights: {len(rejected_ids):,}")
print(f"Docx saved → {docx_path}")
print(f"Stage manifest written → {stage_manifest_path}")
print(f"Pipeline manifest written → {pipeline_manifest_path}")

Accepted insights: 44
Rejected insights: 10
Docx saved → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_124534_miami_geo_group_45fd548b/reports/trend_tracker_report.docx
Stage manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_124534_miami_geo_group_45fd548b/metadata/stage_manifest_03_insights_generation.json
Pipeline manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/miami_geo_group/2026-04-02/20260402_124534_miami_geo_group_45fd548b/metadata/pipeline_manifest.json
